In [6]:
import pandas as pd
import json
import datasets
import os

In [4]:
# Path to the .csv file containing the data
dataset_path = "/projects/iris/rferreir/datasets/GridRoute/original_data/dataset.csv"

# Path to the .csv file containing the best path
reference_path = "/projects/iris/rferreir/datasets/GridRoute/original_data/reference.csv"

# Path to the final json file
output_file = "/projects/iris/rferreir/datasets/GridRoute/dataset.json"

## Build the dataset

In [3]:
dataset = pd.read_csv(dataset_path)
reference = pd.read_csv(reference_path)

In [4]:
import ast
dataset['all_obstacle_coords'] = dataset['all_obstacle_coords'].apply(lambda x : ast.literal_eval(x) if isinstance(x, str) else x)
reference['Path'] = reference['Path'].apply(lambda x : ast.literal_eval(x) if isinstance(x, str) else x)

In [5]:
dataset.head()

,matrix_size,obstacles,all_obstacle_coords,start_x,start_y,end_x,end_y
0,10,"[{'top_left': (7, 6), 'bottom_right': (9, 8)},...","[(7, 6), (7, 7), (7, 8), (8, 6), (8, 7), (8, 8...",5,0,7,9
1,10,"[{'top_left': (7, 6), 'bottom_right': (9, 8)},...","[(7, 6), (7, 7), (7, 8), (8, 6), (8, 7), (8, 8...",9,3,3,1
2,10,"[{'top_left': (7, 6), 'bottom_right': (9, 8)},...","[(7, 6), (7, 7), (7, 8), (8, 6), (8, 7), (8, 8...",3,3,4,2
3,10,"[{'top_left': (7, 6), 'bottom_right': (9, 8)},...","[(7, 6), (7, 7), (7, 8), (8, 6), (8, 7), (8, 8...",4,0,4,6
4,10,"[{'top_left': (7, 6), 'bottom_right': (9, 8)},...","[(7, 6), (7, 7), (7, 8), (8, 6), (8, 7), (8, 8...",1,6,8,5


In [6]:
reference.head()

,Matrix_Size,Start_X,Start_Y,End_X,End_Y,Path,obstacles
0,10,5,0,7,9,"[(5, 0), (5, 1), (5, 2), (5, 3), (5, 4), (5, 5...","[{'top_left': (7, 6), 'bottom_right': (9, 8)},..."
1,10,9,3,3,1,"[(9, 3), (9, 2), (9, 1), (8, 1), (7, 1), (6, 1...","[{'top_left': (7, 6), 'bottom_right': (9, 8)},..."
2,10,3,3,4,2,"[(3, 3), (3, 2), (4, 2)]","[{'top_left': (7, 6), 'bottom_right': (9, 8)},..."
3,10,4,0,4,6,"[(4, 0), (4, 1), (4, 2), (4, 3), (4, 4), (4, 5...","[{'top_left': (7, 6), 'bottom_right': (9, 8)},..."
4,10,1,6,8,5,"[(1, 6), (1, 5), (2, 5), (3, 5), (4, 5), (5, 5...","[{'top_left': (7, 6), 'bottom_right': (9, 8)},..."


In [7]:
final_data = {"matrix_size":[], "start_x":[], "start_y":[], "end_x":[], "end_y":[], "obstacles":[], "path":[]}

In [8]:
obs = dataset['all_obstacle_coords'][0]
print(len(obs))

18


In [9]:
matrix_size = dataset['matrix_size']
startX = dataset['start_x']
startY = dataset['start_y']
endX = dataset['end_x']
endY = dataset['end_y']
obstacles = dataset['all_obstacle_coords']
path = reference['Path']

In [10]:
dics = []
for ms, sX, sY, eX, eY, obs, p in zip(matrix_size, startX, startY, endX, endY, obstacles, path):
    dic = {"matrix_size":ms, 
           "start": [sX, sY],
           "end": [[eX, eY]],
           "obstacles_coords":obs,
           "path":p
          }
    dics.append(dic)

In [11]:
with open(output_file, 'w') as f:
    json.dump(dics, f)

In [12]:
print(len(dics))

300


In [13]:
print(dataset["matrix_size"].value_counts())

matrix_size
10    100
20    100
30    100
Name: count, dtype: int64


## Converting into HF dataset

In [8]:
d1 = datasets.load_dataset('json', data_files={"test":output_file})

print(d1['test'])

# Change the path here if you want the dataset saved locally
#d1.save_to_disk(os.path.join("/projects/iris/rferreir/hub_datasets", "GridRoute"))

Dataset({
    features: ['matrix_size', 'start', 'end', 'obstacles_coords', 'path'],
    num_rows: 300
})


## Sanity check

In [9]:
d2 = datasets.load_dataset("rfr2003/GeoBenchLLM", "GridRoute")

print(d1)
print(d2)

import hashlib
import json
# We check that the content of the dataset is the same
def content_hash(ds):
    h = hashlib.sha256()
    for ex in ds:
        h.update(json.dumps(ex, sort_keys=True).encode())
    return h.hexdigest()

assert content_hash(d1["test"]) == content_hash(d2["test"])

Generating test split: 100%|██████████| 300/300 [00:00<00:00, 26692.08 examples/s]

DatasetDict({
    test: Dataset({
        features: ['matrix_size', 'start', 'end', 'obstacles_coords', 'path'],
        num_rows: 300
    })
})
DatasetDict({
    test: Dataset({
        features: ['matrix_size', 'start', 'end', 'obstacles_coords', 'path'],
        num_rows: 300
    })
})
